In [80]:
"""
Titanic Data Analysis and JSON Export
Author: [Your Name]
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""

# Step 1  - here I have asked the code to import pandas and numpy and changed them to shrot form, as well as setting up the data directory.
import pandas as pd
import numpy as np
import json
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data\titanic.csv


In [81]:
#Step 2

df = pd.read_csv(CSV_FILE)
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

In [82]:
# STEP 3
numeric_columns = df.select_dtypes(include=['number'])

# This stage computes the mean, median, and standard deviation for each numeric column
mean_values = numeric_columns.mean()
median_values = numeric_columns.median()
std_values = numeric_columns.std()

# Display the results in a clear format
print("MEAN")
print(mean_values)

print("MEDIAN")
print(median_values)

print("STANDARD DEVIATION")
print(std_values)

MEAN
PassengerId    446.000000
Survived         0.383838
Pclass           2.308642
Age             29.699118
SibSp            0.523008
Parch            0.381594
Fare            32.204208
dtype: float64
MEDIAN
PassengerId    446.0000
Survived         0.0000
Pclass           3.0000
Age             28.0000
SibSp            0.0000
Parch            0.0000
Fare            14.4542
dtype: float64
STANDARD DEVIATION
PassengerId    257.353842
Survived         0.486592
Pclass           0.836071
Age             14.526497
SibSp            1.102743
Parch            0.806057
Fare            49.693429
dtype: float64


In [83]:
#step 3 sanity check - establishing acceptable ranges
lowest_age = df['Age'].min()
highest_age = df['Age'].max()

print(f"Lowest Age: {lowest_age}")
print(f"=Highest Age: {highest_age}")

lowest_fare = df['Fare'].min()
highest_fare = df['Fare'].max()

print(f"Lowest Fare: {lowest_fare}")
print(f"Highest Fare: {highest_fare}")

Lowest Age: 0.42
=Highest Age: 80.0
Lowest Fare: 0.0
Highest Fare: 512.3292


In [84]:
#step 3 sanity check - checking if results fall within range established in the data set
if 0 <= mean_values['Age'] <= 80:
    print(f"Valid mean age: {mean_values['Age']}")
else:
    print("Warning: Mean age is out of the expected range!")

if 0 <= median_values['Fare'] <= 520:
    print(f"Valid mean age: {median_values['Fare']}")
else:
    print("Warning: Median Fare is out of the expected range!")  


Valid mean age: 29.69911764705882
Valid mean age: 14.4542


In [85]:
# Step 4
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)
 
missing_data = {}
 
for col in df.columns: #loops through each column one by one
    missing_count = df[col].isnull().sum() #here we go through each column one by one (df[col]) and then check in each row if info is missing, returning a boolean result. (.sum) then adds all of the amount of missing rows together
    missing_percent = (missing_count / len(df)) * 100 #here we are creating the percentage of entries missing
    
    # for each colums, open ths logbook we established above and create a new entry that put the amount of missing entries and its percentage together
    missing_data[col] = {
        'count': missing_count, 
        'percentage': missing_percent
    }
    print(f"{col}: {missing_count} missing ({missing_percent:.1f}%)")
    # here we are formatting our work to make it readable. "f" (with quotations marks) allow us to insert our variables, {col} shows column and so on


MISSING VALUES ANALYSIS
PassengerId: 0 missing (0.0%)
Survived: 0 missing (0.0%)
Pclass: 0 missing (0.0%)
Name: 0 missing (0.0%)
Sex: 0 missing (0.0%)
Age: 177 missing (19.9%)
SibSp: 0 missing (0.0%)
Parch: 0 missing (0.0%)
Ticket: 0 missing (0.0%)
Fare: 0 missing (0.0%)
Cabin: 687 missing (77.1%)
Embarked: 2 missing (0.2%)


In [86]:
# STEP 5

# Create a copy of the dataframe for feature engineering
df_features = df.copy()
 
# Feature 1: Family Size (SibSp + Parch + 1 for the passenger themself)
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1
# first component df_features creates a new column. SibSp (sibling and spouse count in the df row), and adds it to the Parch (parents and children), the +1 is for the guest in that row
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))
#this prints the selected columbs and .head10 is a panda function that stops is printing all 891 entries. Proves code is working without flooding the screen
 
# Feature 2: Is Alone (True if FamilySize is 1, False otherwise)
df_features['IsAlone'] = (df_features['FamilySize'] == 1)
print(df_features[['FamilySize', 'IsAlone']].head(10))
 #again we creates a new column changing family size being 1 into a boolean output for analysis. 
# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'
 #a function that is classifying age ranges into a string

# Apply the function across the Age column
df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))

#creates a new column 'agegroup' and applys the function we just made above to the age column to fill it.
 
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50) #blankline for space and 50 = signs for formatiing
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)
 
# Example to get you started:
print("\nFamily Size by Survival:")
#title
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
# creates the variable family_survival. Splits the df by survived (1) or not (0), and then focuses on the family size
print(family_survival)
 #this will show the size of the families that survived or did not survive to see if this was a factor in analysis.

# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)
 
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]
 
print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2    False
1           2    False
2           1     True
3           2    False
4           1     True
5           1     True
6           1     True
7           5    False
8           3    False
9           2    False
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0 

In [90]:
#step 5 - Additional Feature
df_features['HasCabin'] = df_features['Cabin'].notnull()

#was getting an error until i added the below, apparently  i needed to redo this breakdown so the new column could use them in the equation
""
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]

print(df_features[['Cabin', 'HasCabin']].head(10))

print("\nCabin Recorded by Survival:")
cabin_survival = df_features.groupby('Survived')['HasCabin'].agg(['mean', 'median', 'std'])*100
print(cabin_survival)

print("\nHas Cabin:")
survived_cabin_pct = survived['HasCabin'].mean() * 100
not_survived_cabin_pct = not_survived['HasCabin'].mean() * 100

print(f"  Survived mean: {survived_cabin_pct:.1f}%")
print(f"  Not Survived mean: {not_survived_cabin_pct:.1f}%")
print(f"  Difference: {abs(survived_cabin_pct - not_survived_cabin_pct):.1f}%")


  Cabin  HasCabin
0   NaN     False
1   C85      True
2   NaN     False
3  C123      True
4   NaN     False
5   NaN     False
6   E46      True
7   NaN     False
8   NaN     False
9   NaN     False

Cabin Recorded by Survival:
               mean  median        std
Survived                              
0         12.386157     0.0  32.972399
1         39.766082     0.0  49.013173

Has Cabin:
  Survived mean: 39.8%
  Not Survived mean: 12.4%
  Difference: 27.4%


# this seemed like a much better point of analysis as the difference is wide enough to show a positive correlation bewtween having a cabin and survival

In [88]:
#Step 6

import json
from datetime import datetime
import pandas as pd

class Passenger:
    """
    Represents a passenger with all their information
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None, title=None, has_cabin=None):
        # __init__ stands for initialize and is a python constructer, self refers to the specific passenger/guest in question, the rest are the required fields. the =none just fills the row if there is not data included to stop an error
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
         #first we set up the variables listed above one by one. We turn the passenger id into an integer using (int) pd.notna check if the ID is present, if not it prints "none" to avoid crash
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = bool(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None
        self.has_cabin = bool(has_cabin) if pd.notna(has_cabin) else None
    
    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        #to_dict is just a way of translating our custom made object into something JSON can read and use, ready for transfer and unpacking elsewhere
        return {
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title,
            'has_cabin': self.has_cabin
        }

class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe # allows the functions in this class to access the raw data easily
        self.passengers = [] # the creates an empty list ready to receive the passanger object
        self._create_passengers()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe rows."""
        for idx, row in self.dataframe.iterrows():
            passenger = Passenger(
                passenger_id=row.get('PassengerId', idx),
                name=row.get('Name', None),
                age=row.get('Age', None),
                sex=row.get('Sex', None),
                survived=row.get('Survived', None),
                pclass=row.get('Pclass', None),
                fare=row.get('Fare', None),
                embarked=row.get('Embarked', None),
                family_size=row.get('FamilySize', None),
                is_alone=row.get('IsAlone', None),
                title=row.get('Title', None),
                has_cabin=row.get('HasCabin', None)
            )
            self.passengers.append(passenger)
    
    def get_summary_stats(self):
        total = len(self.passengers)
        survived_count = sum(1 for p in self.passengers if p.survived == 1)
        did_not_survive_count = sum(1 for p in self.passengers if p.survived == 0)
        
        valid_ages = [p.age for p in self.passengers if p.age is not None]
        avg_age = sum(valid_ages) / len(valid_ages) if valid_ages else 0.0
        
        valid_fares = [p.fare for p in self.passengers if p.fare is not None]
        avg_fare = sum(valid_fares) / len(valid_fares) if valid_fares else 0.0
        
        return {
            'total_passengers': total,
            'survived': survived_count,
            'did_not_survive': did_not_survive_count,
            'average_age': round(avg_age, 2),
            'average_fare': round(avg_fare, 2)
        }

    def to_json(self, filename='titanic_data.json'):
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': float(self.dataframe['Survived'].mean()) if 'Survived' in self.dataframe else None
            },
            'passengers': [p.to_dict() for p in self.passengers]
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2)
        
        print(f"\n[Success] Data successfully exported to {filename}")
        return data



print("CHECKING IF READY FOR EXPORT")


dataset = TitanicDataset(df_features)
print(f"1. Class Loading Data: {len(dataset.passengers)} passenger objects created.")

stats = dataset.get_summary_stats()
print("2 & 3. Statistics Calculated & Missing Values analyzed:")
for key, value in stats.items():
    print(f"   - {key.replace('_', ' ').title()}: {value}")
    
sample_p = dataset.passengers[0]
print(f"4. Feature Engineering Check: FamilySize={sample_p.family_size}, HasCabin={sample_p.has_cabin}")

exported_data = dataset.to_json('titanic_data.json')
assert 'metadata' in exported_data
assert 'passengers' in exported_data
print("5 & 6. JSON File Created & Validation Passes successful!")


CHECKING IF READY FOR EXPORT
1. Class Loading Data: 891 passenger objects created.
2 & 3. Statistics Calculated & Missing Values analyzed:
   - Total Passengers: 891
   - Survived: 342
   - Did Not Survive: 549
   - Average Age: 29.7
   - Average Fare: 32.2
4. Feature Engineering Check: FamilySize=2, HasCabin=False

[Success] Data successfully exported to titanic_data.json
5 & 6. JSON File Created & Validation Passes successful!


In [89]:
#Step 7
import json

JSON_FILE = 'titanic_data.json'

with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

    print(f"1. Top-Level Keys Found: {list(json_data.keys())}")

# 2. Verify Metadata content
metadata = json_data.get('metadata', {})
print(f"2. Metadata Inspection:")
for key, val in metadata.items():
    print(f"   - {key}: {val}")

# 3. Verify Passenger Data presence & count
passengers = json_data.get('passengers', [])
print(f"3. Passenger Records: {len(passengers)} records loaded successfully.")

# 4. Check a sample record to ensure fields and types are correct
if passengers:
    sample_passenger = passengers[0]
    print(f"4. Sample Passenger Record (ID: {sample_passenger.get('passenger_id')}):")
    print(f"   - Name: {sample_passenger.get('name')}")
    print(f"   - Age: {sample_passenger.get('age')}")
    print(f"   - Survival: {sample_passenger.get('survived')}")
    print(f"   - Family Size: {sample_passenger.get('family_size')}")
    print(f"   - Has Cabin: {sample_passenger.get('has_cabin')}")

print("="*50)
print("Validation Complete: JSON file is valid, parsable, and complete!")
print("="*50)

1. Top-Level Keys Found: ['metadata', 'passengers']
2. Metadata Inspection:
   - dataset_name: Titanic Passenger Dataset
   - export_date: 2026-10-31T22:15:46.566172
   - total_passengers: 891
   - survival_rate: 0.3838383838383838
3. Passenger Records: 891 records loaded successfully.
4. Sample Passenger Record (ID: 1):
   - Name: Braund, Mr. Owen Harris
   - Age: 22.0
   - Survival: 0
   - Family Size: 2
   - Has Cabin: False
Validation Complete: JSON file is valid, parsable, and complete!


In [ ]:
# VISUAL COMPARISON: SURVIVED, NOT SURVIVED, AND DIFFERENCE
import matplotlib.pyplot as plt

# Calculate the mean FamilySize for each survival group.
family_means = df_features.groupby('Survived')['FamilySize'].mean()
family_values = pd.Series({
    'Survived': family_means.get(1, 0),
    'Not survived': family_means.get(0, 0),
    'Difference': abs(family_means.get(1, 0) - family_means.get(0, 0))
})

# Calculate the percentage with a recorded cabin for each survival group.
cabin_means = df_features.groupby('Survived')['HasCabin'].mean() * 100
cabin_values = pd.Series({
    'Survived': cabin_means.get(1, 0),
    'Not survived': cabin_means.get(0, 0),
    'Difference': abs(cabin_means.get(1, 0) - cabin_means.get(0, 0))
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2a9d8f', '#e76f51', '#457b9d']

charts = [
    (axes[0], family_values, 'FamilySize', 'Average family size', '{:.2f}'),
    (axes[1], cabin_values, 'HasCabin', 'Passengers with a recorded cabin', '{:.1f}%')
]

for ax, values, title, ylabel, label_format in charts:
    bars = ax.bar(values.index, values.values, color=colors, width=0.65)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=0)
    for bar, value in zip(bars, values.values):
        ax.annotate(label_format.format(value),
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha='center', va='bottom', xytext=(0, 4),
                    textcoords='offset points')

axes[0].set_ylim(0, max(family_values) * 1.25)
axes[1].set_ylim(0, 100)
fig.suptitle('Titanic Feature Comparison by Survival Outcome', fontsize=16)
fig.tight_layout()
plt.show()